# Echo state networks: student quickstart

This notebook builds a conventional input-driven echo state network (ESN),
trains only its linear readout, and checks more than whether one attractor plot
looks plausible.

Related notebooks:

- `Tutorial_ABM.ipynb`: agent-based swarm dynamics;
- `Tutorial_swarmRC.ipynb`: replace the numerical ESN reservoir with a swarm;
- `advanced/Existing_Model_Validation.ipynb`: Jaeger and other published-model checks.

In [ ]:
mkpath("FIGURES/ESN")

## 1. Define the ESN pipeline

| component | choice here | alternatives |
|---|---|---|
| input | three Lorenz coordinates | any consistently sampled numeric signal |
| reservoir | sparse leaky `tanh` network | change size, spectral radius, input scale, leak and density |
| observation | complete ESN state | a reduced or transformed state in a new experiment |
| readout | affine ridge regression | change regularisation or target |
| task | next-state prediction and autonomous rollout | forecast horizons, classification or control |

Teacher forcing supplies the true recent input. Free running feeds predictions
back as input. A model can perform well under teacher forcing and still fail
quickly in free run.

In [ ]:
include("RC/my_reservoir_core.jl")
include("RC/my_esn.jl")
include("RC/my_reservoir_report.jl")
include("TIME_SERIES/my_systems.jl")

using Random

## 2. Choose and inspect the input

Lorenz is useful here because its three-dimensional state is known and its
switching dynamics make autonomous prediction non-trivial. Replace
`driver=:lorenz` with `:noise` as a sanity check: noise has no deterministic
attractor for an ESN to learn.

In [ ]:
driver = :lorenz
rng_data = MersenneTwister(1)
lor = lorenz_data(rng=rng_data)
noise_data = randn(MersenneTwister(99), size(lor.data)...)

data_raw = driver == :lorenz ? lor.data : noise_data
labels_esn = driver == :lorenz ? lor.labels : ["u$(i)" for i in axes(noise_data, 1)]

# Standardise each channel (zero mean, unit variance) before it ever reaches
# the reservoir. Raw Lorenz z ranges up to ~50; with input_scale=0.2 below,
# that alone is a pre-activation contribution of ~10 -- deep in tanh's
# saturated regime for most units. See the note after Section 5 for what
# that saturation actually does to free-run prediction, with numbers.
μ_in, σ_in = feature_stats(data_raw)
data_esn = standardise!(copy(data_raw), μ_in, σ_in)

fig_input = Figure(size=(950, 330))
ax1 = Axis(fig_input[1, 1], xlabel="sample", ylabel=labels_esn[1],
    title="$(driver) input (standardised)")
lines!(ax1, data_esn[1, 1:800])
ax2 = Axis3(fig_input[1, 2], xlabel=labels_esn[1], ylabel=labels_esn[2],
    zlabel=labels_esn[3], title="state-space trajectory")
lines!(ax2, data_esn[1, 1:2500], data_esn[2, 1:2500], data_esn[3, 1:2500])
fig_input

## 3. Build and train the reservoir

The recurrent and input weights are fixed after construction. Training fits
the ridge readout only. The values below are transparent starting values, not
universally optimal hyperparameters.

In [ ]:
esn_choice = (
    units=300,
    spectral_radius=0.9,
    input_scale=0.2,
    leak=0.3,
    density=0.05,
    ridge=1.0,
)

res_esn = BasicESN(
    size(data_esn, 1);
    Nh=esn_choice.units,
    spectral_radius=esn_choice.spectral_radius,
    input_scale=esn_choice.input_scale,
    leak=esn_choice.leak,
    density=esn_choice.density,
    rng=MersenneTwister(2),
)

In [ ]:
separability_sequences = [
    data_esn[:, 1:700],
    data_esn[:, 901:1600],
    data_esn[:, 1801:2500],
]

out_esn = train_and_evaluate_reservoir(
    res_esn;
    data=data_esn,
    shift=300, train_len=3000, predict_len=700,
    washout=150, ridgeλ=esn_choice.ridge,
    seed=1, rng=MersenneTwister(3),
    labels=labels_esn, dt_data=lor.dt_data,
    separability_sequences=separability_sequences,
    compute_reservoir_diagnostics=true,
    memory_maxlag=50, memory_input_dim=1,
);

## 4. Inspect prediction and reservoir diagnostics

The held-out prediction shows whether the trained readout generalises. The attractor plot shows the geometry of the predicted variables; it does not establish long-term trajectory agreement.

In [ ]:
print_metrics("Teacher-forced", out_esn.m_tf)
print_metrics("Free-run", out_esn.m_fr)

plot_truth_vs_pred_2d(
    out_esn.Ytest[1:2, :], out_esn.Ygen[1:2, :];
    washout=0, dt=lor.dt_data, labels=labels_esn[1:2],
    title="ESN free run: truth and generated trajectory",
)

## 5. Validate the reservoir

Validation asks whether the internal representation is usable:

- **memory:** can a linear readout recover delayed inputs?
- **stability:** do small state perturbations decay, persist or explode?
- **separability:** do distinct input sequences produce distinguishable states?
- **feature health:** how many effective directions does the state matrix use?

There is no single universally correct curve. Interpret these checks relative
to the task and compare them when changing ESN hyperparameters.

In [ ]:
display(plot_memory_curve(out_esn.memory_diag))
display(plot_stability_curve(out_esn.stability_diag; dt=lor.dt_data))
display(plot_separability_matrix(out_esn.separability_diag))

state_health = state_matrix_summary(out_esn.Xtrain)
state_health

### How to read these diagnostics

- **Memory curve:** $R^2$ for reconstructing the input from $k$ steps earlier. A useful reservoir retains recent inputs and gradually forgets older ones.
- **Stability curve:** distance between two copies after one is perturbed. Decay indicates that the shared input restores a common state; persistent growth indicates instability.
- **Separability matrix:** distances between states produced by different inputs. Larger off-diagonal values mean the readout can distinguish those input histories.
- **Feature health:** `frac_near_pm1` measures saturation; `participation_ratio` and `effective_rank` measure how many independent directions are used; `gram_condition` warns when fitting the readout is numerically fragile.

Inspect these together. Strong prediction with saturated, low-rank or unstable states may not survive autonomous use.

## 6. Make one controlled change

Try one change at a time and rerun from the ESN construction cell:

- lower `leak` to lengthen the reservoir timescale;
- vary spectral radius to change recurrent amplification;
- vary input scale to avoid a nearly linear or fully saturated response;
- vary ridge regularisation and compare held-out prediction;
- switch the driver to noise and check which apparent successes disappear.

Use multiple random seeds before claiming that an architecture is better.
The Jaeger benchmark and quantitative memory-capacity reproduction belong in
`advanced/Existing_Model_Validation.ipynb`, where they serve as implementation checks
rather than as the basic student workflow.

## Where to go next

The interface used here—`reset!`, `reservoir_step!`, `feature_map`, feature
collection and a trained linear readout—is also used by
`Tutorial_swarmRC.ipynb`. That is the key comparison: the reservoir dynamics
and observation layer change, while the training machinery remains familiar.